# 03 - Benchmark models

Mean, naive, daily/weekly seasonal naive and drift (Part 3), evaluated with rolling-origin 24 h forecasts over the final 14 days.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from appliance_energy.config import *
from appliance_energy import data, eda, features, evaluation, plotting
from appliance_energy.models import benchmarks as bm

In [ ]:
hourly = data.resample_hourly(data.load_raw())
y = hourly[TARGET]
train, test = data.train_test_split(y, TEST_STEPS)
rows = []
fc_df = pd.DataFrame({'actual': test})
for name, fn in bm.BENCHMARKS.items():
    fc = bm.rolling_origin_forecast(y, test.index, fn, HORIZON)
    fc_df[name] = fc
    rows.append(evaluation.evaluate_forecast(name, test, fc, train))
evaluation.metrics_table(rows)

In [ ]:
plotting.plot_forecast_comparison(
    train, test, fc_df.drop(columns='actual'),
    'fig08_benchmarks.png', 'Benchmark forecasts')
display(Image(str(FIGURE_DIR / 'fig08_benchmarks.png')))

The weekly seasonal naive has the best benchmark MAE (it captures both the daily shape and weekend effects), while the flat mean wins on RMSE because copying last week also copies its unrepeatable spikes. Naive and drift are far worse - appliance use mean-reverts within hours, so the last observed value carries little information about tomorrow.